In [3]:
#Xác định n mốc nội suy tối ưu trên [a, b]
"""
    optimal_nodes!(x::AbstractVector{T}, a::Real, b::Real; endpoints::Symbol=:none) where T<:AbstractFloat
    Ghi đè n mốc nội suy tối ưu trên [a,b] vào vector x(đã được sắp tăng dần)
    Các tuỳ chọn cho endpoints:
    - :none : Mốc Gauss_Chebyshev(nghiệm T_n, ko chứa 2 đầu mút)
    - :both : Mốc Chebyshev-Lobatto(chứa cả a và b)
    - :left : Mốc Chebyshev-Radau trái(chứa a)
    - :right : Mốc Chebyshev-Radau phải(chứa b)
"""
function optimal_nodes!(x::AbstractVector{T}, a::Real, b::Real; endpoints::Symbol=:none) where T<: AbstractFloat
    n=length(x)
    n>=1 || throw(ArgumentError("Số mốc n phải >=1"))
    if endpoints === :both && n< 2
        throw(ArgumentError("Nếu lấy cả 2 đầu thì số mốc phải >=2"))
    end
    mid_val = T(0.5) * T(a+b)
    half_len = T(0.5) * T(b-a)
    half_n = div(n,2)
    if endpoints === :none
        @inbounds for k in 1:half_n
            theta = T(2k-1)*T(pi)/T(2n)
            t_k = -cos(theta)
            x[k] = muladd(half_len, t_k, mid_val)
            x[n-k+1] = muladd(half_len, -t_k, mid_val)
        end
        if isodd(n)
            @inbounds x[half_n+1] = mid_val
        end
    elseif endpoints === :both
        @inbounds for k in 1:half_n
            theta = T(k-1)*T(pi)/T(n-1)
            t_k = -cos(theta)
            x[k] = muladd(half_len, t_k, mid_val)
            x[n-k+1] = muladd(half_len, -t_k, mid_val)
        end
        if isodd(n)
            @inbounds x[half_n+1] = mid_val
        end
    elseif endpoints === :left
        @inbounds x[1] = T(a)
        denom = T(2n-1)
        @inbounds for k in 2:n
            theta = T(2*(k-1))*T(pi)/denom
            t_k = -cos(theta)
            x[k] = muladd(half_len, t_k, mid_val)
        end
    elseif endpoints === :right
        denom = T(2n-1)
        @inbounds for k in 1:(n-1)
            theta = T(2*(n-k))*T(pi)/denom
            t_k = cos(theta)
            x[k] = muladd(half_len, t_k, mid_val)
        end
        @inbounds x[n] = T(b)
    else
        throw(ArgumentError("endpoints không hợp lệ. Chọn :none, :both, :left hoặc :right"))
    end
    return x
end
function optimal_nodes(a::Real, b::Real, n::Integer; endpoints::Symbol=:none, T::Type=Float64)
    x=Vector{T}(undef, n)
    return optimal_nodes!(x, a, b; endpoints=endpoints)
end

optimal_nodes (generic function with 1 method)

In [4]:
node_none = optimal_nodes(-2.0, 3.0, 5, endpoints=:none)
println(round.(node_none, digits=32))

[-1.8776412907378839, -0.969463130731183, 0.5, 1.9694631307311827, 2.8776412907378837]
